## RoBERTa binary (clear non-reply) evaluation

**Dataset:** `ailsntua/QEvasion` — **test** split only (308 examples).  
This is **not** part of training data: training uses the **train** split. The test set is held out for evaluation.

Run the save cell below to export the test set as **input** for the training script and Granite notebook (e.g. `dataset/qevasion_test_308.csv`).

In [1]:
import pandas as pd
import huggingface_hub
from datasets import load_dataset

ds = load_dataset("ailsntua/QEvasion")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
# Use all data (not just subset)
train_data = ds['train']
test_data = ds['test']

Roberta - with oversampling

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Path
local_model_path = "."

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(local_model_path)

# Load the model
# Replace AutoModelForSequenceClassification with the appropriate model class if different (e.g., AutoModelForMaskedLM)
model = AutoModelForSequenceClassification.from_pretrained(local_model_path)

# Example of how to use the loaded model and tokenizer:
text = "This is an example sentence to test the model."
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

# Perform inference
with torch.no_grad():
    outputs = model(**inputs)

# You can then process the outputs, e.g., get logits for classification
logits = outputs.logits
predicted_class_id = torch.argmax(logits, dim=-1).item()

print(f"Model and tokenizer loaded successfully from {local_model_path}.")
print(f"Example text: {text}")
print(f"Predicted class ID: {predicted_class_id}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model and tokenizer loaded successfully from ..
Example text: This is an example sentence to test the model.
Predicted class ID: 0


In [4]:
# Add 'is_clear_non_reply_label' column to test_data
def add_is_clear_non_reply_label_column(example):
    example['is_clear_non_reply_label'] = 1 if example['clarity_label'] == 'Clear Non-Reply' else 0
    return example

print("Adding 'is_clear_non_reply_label' column to test data...")
# Apply the function to the test_data
test_data = test_data.map(add_is_clear_non_reply_label_column)
print("Done adding 'is_clear_non_reply_label' column.")

# Display a sample to verify
print("Sample of test_data with new 'is_clear_non_reply_label' column:")
print(test_data[0])

Adding 'is_clear_non_reply_label' column to test data...


Map:   0%|          | 0/308 [00:00<?, ? examples/s]

Done adding 'is_clear_non_reply_label' column.
Sample of test_data with new 'is_clear_non_reply_label' column:
{'title': None, 'date': None, 'president': None, 'url': 'https://www.presidency.ucsb.edu/documents/the-presidents-news-conference-3', 'question_order': 5, 'interview_question': 'Q. What about the redline, sir?', 'interview_answer': "Well, the world has made it clear that these tests caused us to come together and work in the United Nations to send a clear message to the North Korean regime. We're bound up together with a common strategy to solve this issue peacefully through diplomatic means.Kevin [Kevin Corke, NBC News].", 'gpt3.5_summary': None, 'gpt3.5_prediction': None, 'question': ' Inquiring about the status or information regarding the redline.', 'annotator_id': None, 'annotator1': 'Dodging', 'annotator2': 'General', 'annotator3': 'Dodging', 'inaudible': False, 'multiple_questions': False, 'affirmative_questions': True, 'index': 0, 'clarity_label': 'Ambivalent', 'evasio

In [5]:
# Define the tokenization function for the model
# RoBERTa does not use the literal string "[SEP]" the way BERT does.
# The correct way is to pass (question, answer) as a pair to the tokenizer.
def tokenize_function(examples):
    return tokenizer(
        examples["question"],          # first sequence
        examples["interview_answer"],  # second sequence
        truncation=True,               # cut off if too long
        padding="max_length",          # ensures every example has same length
        max_length=256                 # keep your original length for now
    )

print("Tokenizing test data for model prediction...")
# Tokenize test data
tokenized_test_for_prediction = test_data.map(tokenize_function, batched=True)

# The Hugging Face Trainer expects the label column to be named "labels".
# Your dataset column is currently "label".
# Renaming prevents weird bugs / missing-label issues during training and evaluation.
# We are not training here, but consistency helps if labels are present.
if "label" in tokenized_test_for_prediction.column_names:
    tokenized_test_for_prediction = tokenized_test_for_prediction.rename_column("label", "labels")

print("Tokenization complete!")
print(f"Test data for prediction: {len(tokenized_test_for_prediction)} examples")

Tokenizing test data for model prediction...


Map:   0%|          | 0/308 [00:00<?, ? examples/s]

Tokenization complete!
Test data for prediction: 308 examples


In [11]:
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

# Set the model to evaluation mode
model.eval()

# Create a DataLoader for the tokenized test set
# Prepare dataset for PyTorch, only keeping necessary columns for model input
tokenized_test_for_prediction.set_format("torch", columns=["input_ids", "attention_mask"])

test_dataloader = DataLoader(tokenized_test_for_prediction, batch_size=16)

model_predictions = []

print("Making predictions on the test set with the model...")

# Iterate over the test data
for batch in tqdm(test_dataloader):
    # Move batch to appropriate device (CPU or GPU)
    # Assuming 'model' is already on the correct device (e.g., CPU as in your case)
    inputs = {k: v for k, v in batch.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    batch_preds = torch.argmax(logits, dim=-1)

    model_predictions.extend(batch_preds.cpu().numpy())

print("Model predictions complete!")

# Add predictions as a new column to the original test_data, named 'try_3' as requested
test_data = test_data.add_column("try_3", model_predictions)

print("Original test_data with 'is_clear_non_reply_label' and 'try_1' (model prediction) columns:")
print(test_data[0])
print(test_data.column_names)

Making predictions on the test set with the model...


100%|██████████| 20/20 [02:32<00:00,  7.62s/it]

Model predictions complete!
Original test_data with 'is_clear_non_reply_label' and 'try_1' (model prediction) columns:
{'title': None, 'date': None, 'president': None, 'url': 'https://www.presidency.ucsb.edu/documents/the-presidents-news-conference-3', 'question_order': 5, 'interview_question': 'Q. What about the redline, sir?', 'interview_answer': "Well, the world has made it clear that these tests caused us to come together and work in the United Nations to send a clear message to the North Korean regime. We're bound up together with a common strategy to solve this issue peacefully through diplomatic means.Kevin [Kevin Corke, NBC News].", 'gpt3.5_summary': None, 'gpt3.5_prediction': None, 'question': ' Inquiring about the status or information regarding the redline.', 'annotator_id': None, 'annotator1': 'Dodging', 'annotator2': 'General', 'annotator3': 'Dodging', 'inaudible': False, 'multiple_questions': False, 'affirmative_questions': True, 'index': 0, 'clarity_label': 'Ambivalent',

In [12]:
# NOTE: This cell now runs AFTER majority_vote is applied (moved after Cell 9).
# It saves the full RoBERTa predictions with try_1..try_3 + majority_wins.

import pandas as pd
import os

# Convert the Hugging Face Dataset to a pandas DataFrame
test_data_df = test_data.to_pandas()

# Ensure majority column exists (compute if missing)
if "majority_wins" not in test_data_df.columns:
    test_data_df["majority_wins"] = (test_data_df[["try_1", "try_2", "try_3"]].sum(axis=1) >= 2).astype(int)

# Save with predictions for ensemble script
csv_filename = "clear-non-reply-predictions-roberta.csv"
test_data_df.to_csv(csv_filename, index=False)
print(f"'{csv_filename}' saved with {len(test_data_df)} rows (QEvasion test, 308 samples).")

# Also save to dataset/ for training script input
os.makedirs("dataset", exist_ok=True)
input_csv = "dataset/qevasion_test_308.csv"
test_data_df.to_csv(input_csv, index=False)
print(f"Also saved: {input_csv}")

'test_data.csv' has been saved. You can download it from the files sidebar on the left.


In [13]:
def apply_majority_vote(examples):
    all_same = []
    majority_wins = []

    for i in range(len(examples['try_1'])):
        t1 = examples['try_1'][i]
        t2 = examples['try_2'][i]
        t3 = examples['try_3'][i]

        # Check if all are the same
        are_all_same = (t1 == t2 and t2 == t3)
        all_same.append(are_all_same)

        # Determine the majority vote
        # For binary (0/1) with 3 votes, there will always be a clear majority (2-1 split)
        votes_for_0 = 0
        votes_for_1 = 0

        if t1 == 0: votes_for_0 += 1
        else: votes_for_1 += 1
        if t2 == 0: votes_for_0 += 1
        else: votes_for_1 += 1
        if t3 == 0: votes_for_0 += 1
        else: votes_for_1 += 1

        if votes_for_1 > votes_for_0:
            majority = 1
        else:
            majority = 0
        majority_wins.append(majority)

    examples['all_same'] = all_same
    examples['majority_wins'] = majority_wins
    return examples

print("Applying majority vote and checking for agreement across 'try_1', 'try_2', 'try_3'...")
test_data = test_data.map(apply_majority_vote, batched=True)

print("Majority vote applied. Displaying a sample of the updated test_data:")
print(test_data[0])
print(test_data.column_names)

Applying majority vote and checking for agreement across 'try_1', 'try_2', 'try_3'...


Map:   0%|          | 0/308 [00:00<?, ? examples/s]

Majority vote applied. Displaying a sample of the updated test_data:
{'title': None, 'date': None, 'president': None, 'url': 'https://www.presidency.ucsb.edu/documents/the-presidents-news-conference-3', 'question_order': 5, 'interview_question': 'Q. What about the redline, sir?', 'interview_answer': "Well, the world has made it clear that these tests caused us to come together and work in the United Nations to send a clear message to the North Korean regime. We're bound up together with a common strategy to solve this issue peacefully through diplomatic means.Kevin [Kevin Corke, NBC News].", 'gpt3.5_summary': None, 'gpt3.5_prediction': None, 'question': ' Inquiring about the status or information regarding the redline.', 'annotator_id': None, 'annotator1': 'Dodging', 'annotator2': 'General', 'annotator3': 'Dodging', 'inaudible': False, 'multiple_questions': False, 'affirmative_questions': True, 'index': 0, 'clarity_label': 'Ambivalent', 'evasion_label': '', 'is_clear_non_reply_label': 

In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Extract the true labels and predictions
true_labels = test_data['is_clear_non_reply_label']
predictions = test_data['majority_wins']

# Convert to numpy arrays for scikit-learn functions if they aren't already
true_labels_np = np.array(true_labels)
predictions_np = np.array(predictions)

print("Calculating evaluation metrics...")

# Calculate Accuracy
accuracy = accuracy_score(true_labels_np, predictions_np)
print(f"Accuracy: {accuracy:.4f}")

# Calculate Precision
# The 'pos_label' argument is important for binary classification if 1 is the positive class
precision = precision_score(true_labels_np, predictions_np, pos_label=1)
print(f"Precision: {precision:.4f}")

# Calculate Recall
recall = recall_score(true_labels_np, predictions_np, pos_label=1)
print(f"Recall: {recall:.4f}")

# Calculate F1-Score
f1 = f1_score(true_labels_np, predictions_np, pos_label=1)
print(f"F1-Score: {f1:.4f}")

print("Metrics calculated successfully.")

Calculating evaluation metrics...
Accuracy: 0.9221
Precision: 0.4848
Recall: 0.6957
F1-Score: 0.5714
Metrics calculated successfully.
